# Stock-Market Direction Prediction
## Final Six-Model Evaluation and Comparison

**Problem type:** Binary classification  
**Domain:** Finance / stock-market analytics  
**Target:** Predict whether the next closing price increases (`1`) or decreases/remains unchanged (`0`).

This notebook consolidates the outputs of six independently trained models and completes the remaining assessment requirements:

1. Dataset understanding and problem framing
2. Preprocessing and EDA summary
3. Model-design justification
4. Evaluation, cross-validation comparison, prediction demonstration and final-model selection


## AI-tool usage declaration

ChatGPT was used to support explanation, code development, debugging and documentation. Google Colab and Python libraries were used to execute the workflow, generate visualizations, train the models and calculate the reported metrics. All generated code and outputs were reviewed and executed by the project group. AI tools were not used to invent, alter or fabricate experimental results.


## 1. Setup


In [ ]:
from pathlib import Path
import io
import json
import shutil
import warnings
import zipfile

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.metrics import (
    balanced_accuracy_score,
    confusion_matrix,
    matthews_corrcoef,
)

warnings.filterwarnings('ignore')
sns.set_theme(style='whitegrid')
pd.set_option('display.max_colwidth', 100)

OUTPUT_DIR = Path('AIML_Final_Model_Comparison_Results')
PLOT_DIR = OUTPUT_DIR / 'plots'
OUTPUT_DIR.mkdir(exist_ok=True)
PLOT_DIR.mkdir(exist_ok=True)

MODEL_SPECS = [
    {'label': 'Logistic Regression', 'student_id': 'IT25102978', 'token': 'Logistic_Regression_Results'},
    {'label': 'SVM', 'student_id': 'IT25102165', 'token': 'SVM_Results'},
    {'label': 'Decision Tree', 'student_id': 'IT25200808', 'token': 'Decision_Tree_Results'},
    {'label': 'KNN', 'student_id': 'IT25101173', 'token': 'KNN_Results'},
    {'label': 'Random Forest', 'student_id': 'IT25200151', 'token': 'Random_Forest_Results'},
    {'label': 'Gradient Boosting', 'student_id': 'IT25102034', 'token': 'Gradient_Boosting_Results'},
]

print('Setup completed.')


## 2. Upload the six result ZIP files

Upload the six ZIP files produced by the individual model notebooks. The code accepts filenames with additions such as `(1)`.


In [ ]:
def locate_model_archives():
    zip_files = list(Path('.').glob('*.zip'))

    def find_matches(spec):
        return [p for p in zip_files if spec['token'].lower() in p.name.lower()]

    missing = [spec['label'] for spec in MODEL_SPECS if not find_matches(spec)]
    if missing:
        try:
            from google.colab import files
            print('Upload the six model-result ZIP files.')
            files.upload()
            zip_files = list(Path('.').glob('*.zip'))
        except ImportError:
            print('Place the six model-result ZIP files beside this notebook.')

    archives = {}
    for spec in MODEL_SPECS:
        matches = [p for p in zip_files if spec['token'].lower() in p.name.lower()]
        if not matches:
            raise FileNotFoundError(f"Missing result ZIP for {spec['label']}")
        archives[spec['label']] = max(matches, key=lambda p: p.stat().st_mtime)
    return archives


archives = locate_model_archives()
for model, archive in archives.items():
    print(f'{model:20s} -> {archive.name}')


# Step 1: Dataset Understanding and Problem Framing

## Real-world problem

The project addresses a financial forecasting problem: predicting the direction of the next closing price for one of 14 international market indices. This is useful as a decision-support experiment for market monitoring and risk analysis. It must not be interpreted as guaranteed investment advice because market movements depend on many unavailable and unpredictable factors.

## Dataset suitability

- The original dataset contained daily market-index records with `Date`, `Open`, `High`, `Low`, `Close`, `Adj Close`, `Volume` and the categorical market `Index`.
- A supervised target was created separately within each index: `1` when the following closing price was higher than the current closing price, otherwise `0`.
- The dataset is sufficiently large for classical machine learning and includes records from 14 indices.
- The target contains both classes with only moderate imbalance.
- A chronological 70%/15%/15% split was produced separately for each index, ensuring that later observations did not enter an earlier training partition.


In [ ]:
dataset_quality = pd.DataFrame({
    'Property': [
        'Original dataset', 'Market indices', 'Rows containing missing values',
        'Duplicate rows', 'Invalid OHLC records removed', 'Final cleaned dataset',
        'Target 0 records', 'Target 1 records', 'Training partition',
        'Validation partition', 'Testing partition', 'Selected model features'
    ],
    'Value': [
        '112,457 rows × 8 columns', '14', '2,219 (1.97%)', '0', '1',
        '110,223 rows × 10 columns', '51,715', '58,508', '77,150',
        '16,534', '16,539', '5'
    ]
})
display(dataset_quality)


## Final selected features

| Feature | Meaning |
|---|---|
| `Volume_Available` | Binary indicator showing whether a positive volume value was reported |
| `Day` | Standardized calendar day of the month |
| `Index_399001.SZ` | One-hot indicator for the Shenzhen Component Index |
| `Index_IXIC` | One-hot indicator for the NASDAQ Composite Index |
| `Index_TWII` | One-hot indicator for the Taiwan Weighted Index |

The five features were selected using `SelectKBest(f_classif)` fitted only on training data. The target column was excluded from the input features.


In [ ]:
selected_feature_scores = pd.DataFrame({
    'Feature': ['Index_IXIC', 'Day', 'Index_399001.SZ', 'Volume_Available', 'Index_TWII'],
    'F_Score': [42.011222, 14.685516, 9.671010, 8.992051, 6.773950],
    'P_Value': [9.129602e-11, 1.271193e-4, 1.872665e-3, 2.712430e-3, 9.251557e-3]
})
display(selected_feature_scores)


# Step 2: Data Preprocessing and Analysis

The completed group pipeline applied the following operations in a leakage-safe order:

1. Validated the required schema and converted dates.
2. Removed duplicate, missing and invalid financial records.
3. Created the next-day target within each market index.
4. Extracted year, month, day and day-of-week calendar features.
5. Created chronological training, validation and testing partitions.
6. Learned IQR limits from training data and capped outliers in all partitions using those limits.
7. One-hot encoded the market index and aligned columns.
8. Fitted `StandardScaler` only on training data.
9. Fitted `SelectKBest` only on training data and retained the five strongest features.

The original preprocessing notebook contains the detailed missing-value, encoding, outlier, scaling, feature-engineering, feature-selection and target-distribution visualizations.


## 3. Load and validate the six experiment outputs


In [ ]:
def member_name(zip_file, suffix):
    matches = [n for n in zip_file.namelist()
               if n.endswith(suffix) and not n.startswith('__MACOSX/')]
    if len(matches) != 1:
        raise ValueError(f'Expected one {suffix!r}; found {matches}')
    return matches[0]


def read_csv_from_zip(zip_file, suffix):
    return pd.read_csv(io.BytesIO(zip_file.read(member_name(zip_file, suffix))))


experiments = {}
reference_actual = None

for spec in MODEL_SPECS:
    label = spec['label']
    with zipfile.ZipFile(archives[label]) as zf:
        required = [
            'experiment_summary.json', 'final_test_metrics.csv',
            'cross_validation_scores.csv', 'validation_model_comparison.csv',
            'test_predictions.csv'
        ]
        for filename in required:
            member_name(zf, filename)

        summary = json.loads(zf.read(member_name(zf, 'experiment_summary.json')))
        metrics_df = read_csv_from_zip(zf, 'final_test_metrics.csv')
        cv_df = read_csv_from_zip(zf, 'cross_validation_scores.csv')
        validation_df = read_csv_from_zip(zf, 'validation_model_comparison.csv')
        predictions_df = read_csv_from_zip(zf, 'test_predictions.csv')
        target_distribution_df = read_csv_from_zip(zf, 'target_distribution.csv')

    required_prediction_columns = {
        'Actual_Target', 'Predicted_Target', 'Prediction_Correct'
    }
    assert required_prediction_columns.issubset(predictions_df.columns)
    assert len(predictions_df) == 16539

    actual = predictions_df['Actual_Target'].to_numpy()
    if reference_actual is None:
        reference_actual = actual
    else:
        assert np.array_equal(reference_actual, actual), 'Test targets differ between models.'

    experiments[label] = {
        'summary': summary,
        'metrics': metrics_df.iloc[0],
        'cv': cv_df,
        'validation': validation_df,
        'predictions': predictions_df,
        'target_distribution': target_distribution_df,
    }

print('Validated all six archives and confirmed identical test targets.')


## 4. Target distribution and EDA insights


In [ ]:
target_distribution = experiments['KNN']['target_distribution'].copy()
display(target_distribution)

plot_data = target_distribution.set_index('Dataset')[['Down_or_Not_Up_0', 'Up_1']]
ax = plot_data.plot(kind='bar', figsize=(9, 5), color=['#d95f5f', '#4c9f70'])
ax.set_title('Target Distribution Across Chronological Partitions')
ax.set_xlabel('Dataset partition')
ax.set_ylabel('Number of observations')
ax.tick_params(axis='x', rotation=0)
plt.tight_layout()
plt.savefig(PLOT_DIR / '01_target_distribution.png', dpi=200, bbox_inches='tight')
plt.show()


### EDA insights

- Class `1` is moderately more common than Class `0`, increasing from 52.72% in training to 54.56% in testing.
- This temporal change means raw accuracy must be compared with the 54.56% majority-class baseline.
- Approximately 1.97% of original rows contained missing required values, while no exact duplicates were found.
- Zero volume occurred frequently and can represent an index that did not report volume; therefore, it was retained and represented using `Volume_Available`.
- The selected features have statistically detectable but weak relationships with the target. This helps explain why all model ROC-AUC values remain close to 0.50.


# Step 3: Model Design and Implementation

All models were implemented as binary classifiers using Python and scikit-learn. Each individual notebook contains baseline training, five-fold stratified cross-validation, hyperparameter tuning using `GridSearchCV`, validation comparison, final fitting on combined training and validation data, and one final test evaluation.

| Model | Suitability and purpose |
|---|---|
| Logistic Regression | Interpretable probabilistic baseline for a binary target |
| Linear SVM | Margin-based classifier suitable for scaled numerical features |
| Decision Tree | Interpretable model capable of non-linear decision rules |
| KNN | Non-parametric model that captures local neighbourhood patterns |
| Random Forest | Bagged tree ensemble designed to reduce single-tree variance |
| Gradient Boosting | Sequential tree ensemble designed to correct earlier errors |


In [ ]:
design_rows = []
for spec in MODEL_SPECS:
    label = spec['label']
    summary = experiments[label]['summary']
    design_rows.append({
        'Model': label,
        'Student_ID': spec['student_id'],
        'Algorithm': summary['algorithm'],
        'Best_Parameters': json.dumps(summary.get('best_parameters', {})),
        'Best_CV_F1_from_Tuning': summary.get('best_cross_validation_f1', np.nan),
    })

model_design_summary = pd.DataFrame(design_rows)
display(model_design_summary)
model_design_summary.to_csv(OUTPUT_DIR / 'model_design_summary.csv', index=False)


# Step 4: Model Evaluation and Comparison

Accuracy, precision, recall and F1-score are reported, but they must be interpreted together with ROC-AUC, specificity, balanced accuracy, confusion matrices and prediction distributions. This is necessary because a classifier that always predicts the majority class obtains 54.56% accuracy without distinguishing the two classes.


In [ ]:
comparison_rows = []
confusion_matrices = {}

for spec in MODEL_SPECS:
    label = spec['label']
    metric = experiments[label]['metrics']
    pred = experiments[label]['predictions']
    y_true = pred['Actual_Target']
    y_pred = pred['Predicted_Target']
    cm = confusion_matrix(y_true, y_pred, labels=[0, 1])
    tn, fp, fn, tp = cm.ravel()
    confusion_matrices[label] = cm

    comparison_rows.append({
        'Model': label,
        'Accuracy': float(metric['Accuracy']),
        'Precision': float(metric['Precision']),
        'Recall': float(metric['Recall']),
        'F1_Score': float(metric['F1_Score']),
        'ROC_AUC': float(metric['ROC_AUC']),
        'Specificity': tn / (tn + fp),
        'Balanced_Accuracy': balanced_accuracy_score(y_true, y_pred),
        'MCC': matthews_corrcoef(y_true, y_pred),
        'Predicted_0': int((y_pred == 0).sum()),
        'Predicted_1': int((y_pred == 1).sum()),
        'Predicts_Both_Classes': bool(y_pred.nunique() == 2),
    })

comparison = pd.DataFrame(comparison_rows)
majority_baseline = float(pd.Series(reference_actual).value_counts(normalize=True).max())
comparison['Accuracy_vs_Majority_Baseline'] = comparison['Accuracy'] - majority_baseline
comparison['ROC_AUC_vs_Random'] = comparison['ROC_AUC'] - 0.5
comparison = comparison.sort_values(
    ['Predicts_Both_Classes', 'Balanced_Accuracy', 'ROC_AUC'],
    ascending=[False, False, False]
).reset_index(drop=True)

print(f'Majority-class accuracy baseline: {majority_baseline:.4f}')
display(comparison.round(4))
comparison.to_csv(OUTPUT_DIR / 'six_model_test_comparison.csv', index=False)


## Metric comparison

The dashed horizontal line represents the majority-class accuracy baseline. A model reaching this line is not necessarily useful if it predicts only Class `1`.


In [ ]:
metric_columns = ['Accuracy', 'F1_Score', 'ROC_AUC', 'Balanced_Accuracy']
ax = comparison.set_index('Model')[metric_columns].plot(
    kind='bar', figsize=(13, 6), ylim=(0, 0.8), colormap='viridis'
)
ax.axhline(majority_baseline, color='#c0392b', linestyle='--', linewidth=2,
           label=f'Majority baseline ({majority_baseline:.3f})')
ax.axhline(0.5, color='gray', linestyle=':', linewidth=1.5, label='Random AUC/balanced accuracy')
ax.set_title('Final Test Metrics for All Six Models')
ax.set_ylabel('Score')
ax.tick_params(axis='x', rotation=25)
ax.legend(loc='upper center', bbox_to_anchor=(0.5, -0.20), ncol=3)
plt.tight_layout()
plt.savefig(PLOT_DIR / '02_six_model_metric_comparison.png', dpi=200, bbox_inches='tight')
plt.show()


## Predicted-class distributions


In [ ]:
class_counts = comparison.set_index('Model')[['Predicted_0', 'Predicted_1']]
ax = class_counts.plot(kind='bar', stacked=True, figsize=(12, 6), color=['#d95f5f', '#4c9f70'])
ax.set_title('Predicted Class Distribution by Model')
ax.set_xlabel('Model')
ax.set_ylabel('Number of test predictions')
ax.tick_params(axis='x', rotation=25)
plt.legend(['Class 0: Down/Not Up', 'Class 1: Up'])
plt.tight_layout()
plt.savefig(PLOT_DIR / '03_predicted_class_distributions.png', dpi=200, bbox_inches='tight')
plt.show()


## Confusion matrices


In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(16, 10))
for ax, spec in zip(axes.ravel(), MODEL_SPECS):
    label = spec['label']
    sns.heatmap(
        confusion_matrices[label], annot=True, fmt='d', cmap='Blues', cbar=False,
        xticklabels=['Down/Not Up', 'Up'], yticklabels=['Down/Not Up', 'Up'], ax=ax
    )
    ax.set_title(label)
    ax.set_xlabel('Predicted class')
    ax.set_ylabel('Actual class')

plt.suptitle('Final Test Confusion Matrices', fontsize=16, y=1.01)
plt.tight_layout()
plt.savefig(PLOT_DIR / '04_all_confusion_matrices.png', dpi=200, bbox_inches='tight')
plt.show()


## Five-fold cross-validation comparison


In [ ]:
cv_rows = []
for spec in MODEL_SPECS:
    label = spec['label']
    cv_df = experiments[label]['cv']
    cv_rows.append({
        'Model': label,
        'Mean_CV_Accuracy': cv_df['Accuracy'].mean(),
        'SD_CV_Accuracy': cv_df['Accuracy'].std(),
        'Mean_CV_F1': cv_df['F1_Score'].mean(),
        'SD_CV_F1': cv_df['F1_Score'].std(),
        'Mean_CV_ROC_AUC': cv_df['ROC_AUC'].mean(),
        'SD_CV_ROC_AUC': cv_df['ROC_AUC'].std(),
    })

cv_comparison = pd.DataFrame(cv_rows).sort_values('Mean_CV_ROC_AUC', ascending=False)
display(cv_comparison.round(4))
cv_comparison.to_csv(OUTPUT_DIR / 'cross_validation_comparison.csv', index=False)

ax = cv_comparison.set_index('Model')[['Mean_CV_Accuracy', 'Mean_CV_F1', 'Mean_CV_ROC_AUC']].plot(
    kind='bar', figsize=(12, 6), ylim=(0, 0.8), colormap='Set2'
)
ax.set_title('Mean Five-Fold Cross-Validation Scores')
ax.set_ylabel('Mean score')
ax.tick_params(axis='x', rotation=25)
plt.tight_layout()
plt.savefig(PLOT_DIR / '05_cross_validation_comparison.png', dpi=200, bbox_inches='tight')
plt.show()


## Final model selection

### Primary model: KNN

KNN is selected as the most meaningful final performer because:

- It is the only model that produced substantial predictions for both classes.
- It achieved the highest test ROC-AUC among the six models.
- It achieved the highest test balanced accuracy.
- Its confusion matrix demonstrates an attempt to identify both upward and downward movements.

### Secondary comparison model: Decision Tree

Decision Tree is retained as the second model for presentation because:

- It achieved the second-highest test ROC-AUC.
- Its decision rules and feature importance are easier to explain than more complex ensembles.
- It provides a useful contrast with KNN's neighbourhood-based approach.

However, Decision Tree predicted only Class `1` on the final test set. It is therefore presented as an interpretable comparison model, not as evidence of strong predictive performance.


## Prediction demonstration for the selected two models


In [ ]:
prediction_views = []
view_columns = ['Actual_Target', 'Predicted_Target', 'Probability_Up', 'Prediction_Correct']

for label in ['KNN', 'Decision Tree']:
    pred = experiments[label]['predictions'].copy()
    feature_columns = [c for c in pred.columns if c not in view_columns]
    sample = pred[feature_columns + view_columns].head(20).copy()
    sample.insert(0, 'Model', label)
    prediction_views.append(sample)
    print(f'{label} — first 20 unseen test predictions')
    display(sample)

selected_predictions = pd.concat(prediction_views, ignore_index=True)
selected_predictions.to_csv(OUTPUT_DIR / 'selected_models_prediction_samples.csv', index=False)


### How to read the prediction table

- `Actual_Target` is the known answer in the held-out test data.
- `Predicted_Target` is the model's predicted class.
- `Probability_Up` is the model's score/probability for Class `1`.
- `Prediction_Correct` is `True` when the prediction matches the actual target.

These predictions were generated only after training and tuning. The test labels were not used to fit the models.


## Probability distributions for the selected models


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=True)
for ax, label, color in zip(axes, ['KNN', 'Decision Tree'], ['#3b82f6', '#f59e0b']):
    pred = experiments[label]['predictions']
    sns.histplot(
        data=pred, x='Probability_Up', hue='Actual_Target', bins=25,
        stat='density', common_norm=False, element='step', ax=ax,
        palette=['#d95f5f', color]
    )
    ax.axvline(0.5, linestyle='--', color='black', linewidth=1)
    ax.set_title(f'{label}: Predicted Probability of Up')
    ax.set_xlabel('Probability / score for Class 1')

plt.tight_layout()
plt.savefig(PLOT_DIR / '06_selected_model_probability_distributions.png', dpi=200, bbox_inches='tight')
plt.show()


## Final conclusion

The six models produced weak predictive discrimination. Logistic Regression, SVM, Decision Tree and Gradient Boosting predicted every test record as Class `1`; Random Forest predicted only 13 records as Class `0`. Their accuracy of approximately 54.56% is explained by the majority-class proportion rather than an ability to separate upward and downward movements.

KNN was the only model that predicted both classes in meaningful quantities and achieved the highest ROC-AUC and balanced accuracy. It was therefore selected as the best observed model. Nevertheless, its ROC-AUC remained close to 0.50 and its balanced accuracy was approximately 0.502, meaning performance was only slightly above random classification.

The honest project finding is that the selected five-feature representation contains limited information for predicting the next closing-price direction. The experiments are still valid because every model followed the same preprocessing, chronological partitions, cross-validation procedure and evaluation rules.

## Limitations and future improvements

- Stock-market direction is noisy and affected by news, economics and investor behaviour not contained in the dataset.
- The selected five features omit direct historical price, return, momentum and volatility indicators.
- Moderate temporal class drift is visible between the training and test partitions.
- F1-based tuning favoured majority-class predictions in several models.
- Future work could evaluate lagged returns, moving averages, volatility, class-sensitive objectives and decision-threshold tuning while maintaining chronological validation.


## Save the final comparison package


In [ ]:
final_summary = {
    'problem': 'Predict next closing-price direction',
    'target': {'0': 'down or unchanged', '1': 'up'},
    'models_compared': comparison['Model'].tolist(),
    'majority_class_accuracy_baseline': majority_baseline,
    'primary_selected_model': 'KNN',
    'secondary_comparison_model': 'Decision Tree',
    'selection_reason': (
        'KNN was the only model to predict both classes in substantial quantities and '
        'had the highest test ROC-AUC and balanced accuracy. Decision Tree was retained '
        'as an interpretable secondary comparison but collapsed to Class 1 on the test set.'
    ),
    'overall_finding': 'All models showed weak discrimination; ROC-AUC values were close to 0.50.',
}

with open(OUTPUT_DIR / 'final_selection_summary.json', 'w') as f:
    json.dump(final_summary, f, indent=2)

report_text = (
    'FINAL MODEL-COMPARISON SUMMARY\n\n'
    'Problem: Predict next closing-price direction (0 = down/not up, 1 = up)\n'
    'Models compared: Logistic Regression, SVM, Decision Tree, KNN, Random Forest, Gradient Boosting\n'
    f'Majority-class accuracy baseline: {majority_baseline:.4f}\n'
    'Primary selected model: KNN\n'
    'Secondary comparison model: Decision Tree\n\n'
    'Finding:\n'
    'KNN was the only model that predicted both classes in substantial quantities and achieved\n'
    'the highest test ROC-AUC and balanced accuracy. Overall predictive discrimination remained\n'
    "weak because every model's ROC-AUC was close to 0.50.\n"
)
(OUTPUT_DIR / 'final_project_summary.txt').write_text(report_text)

zip_path = shutil.make_archive('AIML_Final_Model_Comparison_Results', 'zip', OUTPUT_DIR)
print('Created:', zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except ImportError:
    print('Download the ZIP from the notebook folder:', zip_path)
